In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/nexsus01/hospitel-data-set/sample_submission.csv
/kaggle/input/datasets/nexsus01/hospitel-data-set/data_dictionary.csv
/kaggle/input/datasets/nexsus01/hospitel-data-set/train.csv
/kaggle/input/datasets/nexsus01/hospitel-data-set/test.csv


In [3]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

In [4]:
# ==========================================
# 1. LOAD DATA
# ==========================================
# Base path for your Kaggle dataset
base_path = '/kaggle/input/datasets/nexsus01/hospitel-data-set/'

train = pd.read_csv(base_path + 'train.csv')
test = pd.read_csv(base_path + 'test.csv')
sample_sub = pd.read_csv(base_path + 'sample_submission.csv')
data_dict = pd.read_csv(base_path + 'data_dictionary.csv')

print("✅ Data loaded successfully!\n")

✅ Data loaded successfully!



In [5]:
# ==========================================
# 2. CHECK SHAPES & COLUMNS
# ==========================================
print("--- SHAPES ---")
print(f"Train:           {train.shape}")
print(f"Test:            {test.shape}")
print(f"Sample Sub:      {sample_sub.shape}\n")

print("--- COLUMNS ---")
train_cols = set(train.columns)
test_cols = set(test.columns)

print(f"Columns ONLY in Train: {train_cols - test_cols}")
print(f"Columns ONLY in Test:  {test_cols - train_cols}\n")

--- SHAPES ---
Train:           (7000, 25)
Test:            (3000, 24)
Sample Sub:      (3000, 2)

--- COLUMNS ---
Columns ONLY in Train: {'readmitted_30d'}
Columns ONLY in Test:  set()



In [6]:
# ==========================================
# 3. CHECK TARGET VARIABLE
# ==========================================
target = 'readmitted_30d'
print(f"--- TARGET DISTRIBUTION ({target}) ---")
if target in train.columns:
    counts = train[target].value_counts()
    print(counts)
    baseline_prob = train[target].mean()
    print(f"\n🎯 Baseline probability (Train Mean): {baseline_prob:.6f}")
    print(f"📌 Sample submission constant value:  {sample_sub['readmitted_30d'].iloc[0]:.6f}")
    print("*(If these match closely, the sample submission is just a naive baseline!)*\n")
else:
    print("❌ Target column not found in train!\n")

--- TARGET DISTRIBUTION (readmitted_30d) ---
readmitted_30d
0    6119
1     881
Name: count, dtype: int64

🎯 Baseline probability (Train Mean): 0.125857
📌 Sample submission constant value:  0.125857
*(If these match closely, the sample submission is just a naive baseline!)*



In [7]:
# ==========================================
# 4. CHECK MISSING VALUES
# ==========================================
print("--- MISSING VALUES ---")
missing_train = train.isnull().sum()
missing_test = test.isnull().sum()

missing_df = pd.DataFrame({
    'Train_Missing_%': (missing_train / len(train)) * 100,
    'Test_Missing_%': (missing_test / len(test)) * 100
})

# Filter to only show columns that actually have missing values
missing_df = missing_df[(missing_df['Train_Missing_%'] > 0) | (missing_df['Test_Missing_%'] > 0)]

if missing_df.empty:
    print("No missing values found in either dataset.\n")
else:
    print(missing_df.sort_values(by='Train_Missing_%', ascending=False).round(2))
    print("\n")

--- MISSING VALUES ---
                  Train_Missing_%  Test_Missing_%
creatinine_mg_dl             4.83            7.07
heart_rate_bpm               4.81            6.87
hemoglobin_g_dl              4.63            6.57
sodium_mmol_l                4.44            6.40
systolic_bp_mmhg             4.41            5.90
followup_days                2.33            3.33




In [8]:
# ==========================================
# 5. CHECK DATA TYPES
# ==========================================
print("--- DATA TYPES (TRAIN) ---")
print(train.dtypes.value_counts())
print("\n")

--- DATA TYPES (TRAIN) ---
float64    9
int64      9
object     7
Name: count, dtype: int64




In [9]:
# ==========================================
# 5. CHECK DATA TYPES
# ==========================================
print("--- DATA TYPES (TRAIN) ---")
print(train.dtypes.value_counts())
print("\n")

# ==========================================
# 6. IDENTIFY LEAKAGE RISKS
# ==========================================
print("--- LEAKAGE & ID CHECKS ---")

# Check uniqueness
print(f"Train IDs unique? {train['patient_id'].nunique() == len(train)}")
print(f"Test IDs unique?  {test['patient_id'].nunique() == len(test)}")

# Check overlap (Crucial for leakage)
train_ids = set(train['patient_id'])
test_ids = set(test['patient_id'])
overlap = train_ids.intersection(test_ids)
print(f"Overlap between Train and Test IDs: {len(overlap)} (Must be 0)")

# Check ID prefixes
print(f"Train ID prefix example: {train['patient_id'].iloc[0]}")
print(f"Test ID prefix example:  {test['patient_id'].iloc[0]}")
print("⚠️ WARNING: Never use 'patient_id' as a feature! The TR/TE prefixes will cause massive leakage.\n")

# Check if target accidentally leaked into test
print(f"Target '{target}' in test set? {target in test.columns} (Must be False)\n")

# Check Data Dictionary notes for unstable/sensitive variables
print("--- DATA DICTIONARY TRUSTWORTHY AI NOTES ---")
notes = data_dict[['variable', 'trustworthy_ai_note']].dropna()
for index, row in notes.iterrows():
    print(f"- {row['variable']}: {row['trustworthy_ai_note']}")

print("\n" + "="*60)
print("STEP 1 COMPLETE. Ready for EDA and Preprocessing.")
print("="*60)

--- DATA TYPES (TRAIN) ---
float64    9
int64      9
object     7
Name: count, dtype: int64


--- LEAKAGE & ID CHECKS ---
Train IDs unique? True
Test IDs unique?  True
Overlap between Train and Test IDs: 0 (Must be 0)
Train ID prefix example: TR00001
Test ID prefix example:  TE02169
⚠️ WARNING: Never use 'patient_id' as a feature! The TR/TE prefixes will cause massive leakage.

Target 'readmitted_30d' in test set? False (Must be False)

--- DATA DICTIONARY TRUSTWORTHY AI NOTES ---
- patient_id: No
- age: No
- sex: Sensitive attribute for subgroup audit
- rurality: Context / subgroup audit
- socioeconomic_index: Context / subgroup audit
- hospital_type: Potential transportability factor
- region: Potential transportability factor
- prior_admissions_12m: Predictor
- comorbidity_count: Predictor
- diabetes: Predictor
- hypertension: Predictor
- chronic_kidney_disease: Predictor
- heart_failure: Predictor
- length_of_stay_days: Predictor
- medication_count: Predictor
- missed_appointments_